## Overview

This notebook integrates the completed Demographic, Behavioural Evidence, and Opportunity components into a final mobility representation. For each constrained representative profile, it constructs trip-purpose probabilities and their corresponding temporal and travel-mode distributions. POI destination weights are then incorporated using the general Opportunity Layer and the industry-based refinement component. The integrated workflow is first tested on selected representative profiles before being applied to the full representative profile space.

### 1. Import Libraries

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from copy import deepcopy

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

### 2. Set Up Project Paths

In [3]:
project_root = Path.cwd().parents[1]

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

### 3. Import Main Inputs

In [4]:
representative_profile_path = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_profile_space.parquet"
)

behavioural_evidence_path = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)

opportunity_layer_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "opportunity_layer.json"
)

industry_opportunity_component_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "industry_to_poi_mapping.json"
)

In [5]:
representative_profile_space = pd.read_parquet(
    representative_profile_path
)

with open(
    behavioural_evidence_path,
    "r",
    encoding="utf-8",
) as file:
    assembled_behavioural_evidence = json.load(file)

with open(
    opportunity_layer_path,
    "r",
    encoding="utf-8",
) as file:
    opportunity_layer = json.load(file)

with open(
    industry_opportunity_component_path,
    "r",
    encoding="utf-8",
) as file:
    industry_component = json.load(file)

In [6]:
print(
    "Representative profile space shape:",
    representative_profile_space.shape,
)

print(
    "Behavioural evidence purposes:",
    len(assembled_behavioural_evidence),
)

print(
    "Opportunity Layer entries:",
    len(opportunity_layer),
)

print(
    "Industry component entries:",
    len(industry_component),
)

Representative profile space shape: (2267232, 14)
Behavioural evidence purposes: 14
Opportunity Layer entries: 14
Industry component entries: 19


### 4. Import Main Functions

These functions integrate the Demographic Layer and Behavioural Evidence Layer and apply the relevant NTS constraints.

In [7]:
from src.demographic_behaviour_layers_integration import (
    construct_constrained_behaviour_representation,
    DEMOGRAPHIC_ATTRIBUTES
)

In [8]:
from src.complete_framework import (
    construct_complete_framework,
    display_complete_framework
)

### 5. Layer Integration: Demographic, Behavioural Evidence, Opportunity(Log Transformation, Industry Component)

In [33]:
import random

random_index = random.randint(
    0,
    len(representative_profile_space) - 1,
)

profile_random = representative_profile_space.iloc[random_index]
(
    complete_profile,
    framework_dataframes,
) = construct_complete_framework(
    representative_profile=profile_random,
    assembled_behavioural_evidence=(
        assembled_behavioural_evidence
    ),
    opportunity_layer=opportunity_layer,
    industry_component=industry_component,
)

display_complete_framework(framework_dataframes)

## Demographic Layer

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership
0,65-74,Female,Lone parent household: All children non-dependent,Economically active and a full-time student: In employment,B Mining and quarrying,"Black, Black British, Black Welsh, Caribbean or African: Other Black",10km to less than 30km,2 cars or vans in household


## Behavioural Evidence Layer
### Trip-purpose Probabilities

,Trip Purpose,Probability
0,Commuting,0.0457
1,Business,0.0131
2,Education,0.0004
3,Escort Education,0.0212
4,Shopping,0.2850
5,Other Escort,0.0472
6,Personal Business,0.1417
7,Visiting Friends at Private Home,0.1092
8,Visiting Friends Elsewhere,0.0709
9,Entertainment or Public Activity,0.0754


### Temporal Distributions

,Trip Purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%)
0,Commuting,0.0024,0.0015,0.0011,0.0022,0.0083,0.0306,0.0797,0.1526,0.1333,0.0366,0.0183,0.0163,0.0219,0.0314,0.0349,0.0538,0.1038,0.1326,0.0628,0.0254,0.0176,0.0113,0.0140,0.0077
1,Business,0.0004,0.0002,0.0006,0.0008,0.0022,0.0107,0.0459,0.0870,0.1127,0.0747,0.0601,0.0756,0.0655,0.0663,0.0634,0.0716,0.0912,0.0746,0.0399,0.0192,0.0152,0.0113,0.0070,0.0038
2,Education,0.0000,0.0000,0.0000,0.0003,0.0000,0.0001,0.0025,0.0980,0.3776,0.0176,0.0097,0.0116,0.0147,0.0084,0.0280,0.3328,0.0548,0.0267,0.0077,0.0047,0.0021,0.0019,0.0005,0.0002
3,Escort Education,0.0000,0.0000,0.0001,0.0001,0.0000,0.0002,0.0018,0.0515,0.3871,0.0341,0.0038,0.0121,0.0109,0.0052,0.0930,0.3096,0.0521,0.0288,0.0053,0.0026,0.0010,0.0002,0.0003,0.0001
4,Shopping,0.0002,0.0000,0.0000,0.0000,0.0000,0.0005,0.0034,0.0129,0.0318,0.0725,0.1179,0.1245,0.1112,0.0937,0.0871,0.0757,0.0727,0.0647,0.0572,0.0420,0.0213,0.0076,0.0022,0.0009
5,Other Escort,0.0007,0.0002,0.0002,0.0001,0.0009,0.0028,0.0092,0.0434,0.0933,0.0741,0.0768,0.0746,0.0814,0.0702,0.0676,0.0833,0.0902,0.0858,0.0594,0.0393,0.0237,0.0115,0.0076,0.0037
6,Personal Business,0.0007,0.0002,0.0002,0.0001,0.0009,0.0028,0.0092,0.0434,0.0933,0.0741,0.0768,0.0746,0.0814,0.0702,0.0676,0.0833,0.0902,0.0858,0.0594,0.0393,0.0237,0.0115,0.0076,0.0037
7,Visiting Friends at Private Home,0.0012,0.0005,0.0003,0.0003,0.0005,0.0028,0.0074,0.0156,0.0279,0.0479,0.0571,0.0631,0.0691,0.0622,0.0607,0.0723,0.0760,0.0936,0.1081,0.0879,0.0552,0.0448,0.0293,0.0162
8,Visiting Friends Elsewhere,0.0012,0.0005,0.0003,0.0003,0.0005,0.0028,0.0074,0.0156,0.0279,0.0479,0.0571,0.0631,0.0691,0.0622,0.0607,0.0723,0.0760,0.0936,0.1081,0.0879,0.0552,0.0448,0.0293,0.0162
9,Entertainment or Public Activity,0.0012,0.0005,0.0003,0.0003,0.0005,0.0028,0.0074,0.0156,0.0279,0.0479,0.0571,0.0631,0.0691,0.0622,0.0607,0.0723,0.0760,0.0936,0.1081,0.0879,0.0552,0.0448,0.0293,0.0162


### Travel-mode Distributions

,Trip Purpose,Car or van driver,Car or van passenger,Motorcycle,Non-local bus,Other Private Transport,Other Public Transport,Other local bus,Pedal Cycle,Surface Rail,Taxi or minicab,Walk
0,Commuting,0.5431,0.0741,0.0093,0.0004,0.0046,0.0411,0.0791,0.0419,0.0693,0.0105,0.1266
1,Business,0.6862,0.0580,0.0018,0.0000,0.0074,0.0238,0.0371,0.0185,0.0684,0.0118,0.0868
2,Education,0.2340,0.2316,0.0002,0.0000,0.0257,0.0071,0.0672,0.0117,0.0116,0.0062,0.4047
3,Escort Education,0.2340,0.2316,0.0002,0.0000,0.0257,0.0071,0.0672,0.0117,0.0116,0.0062,0.4047
4,Shopping,0.4573,0.1774,0.0012,0.0000,0.0047,0.0047,0.0515,0.0089,0.0076,0.0051,0.2815
5,Other Escort,0.5640,0.2826,0.0002,0.0000,0.0076,0.0018,0.0180,0.0021,0.0032,0.0045,0.1160
6,Personal Business,0.4255,0.2229,0.0008,0.0002,0.0081,0.0085,0.0478,0.0096,0.0116,0.0147,0.2503
7,Visiting Friends at Private Home,0.3796,0.3228,0.0023,0.0013,0.0089,0.0148,0.0372,0.0246,0.0292,0.0165,0.1628
8,Visiting Friends Elsewhere,0.3796,0.3228,0.0023,0.0013,0.0089,0.0148,0.0372,0.0246,0.0292,0.0165,0.1628
9,Entertainment or Public Activity,0.3796,0.3228,0.0023,0.0013,0.0089,0.0148,0.0372,0.0246,0.0292,0.0165,0.1628


## Opportunity Layer
### Final POI Mapping

,Trip Purpose,POI Category,Weight
0,Commuting,Employment,1.0000
1,Business,Employment,1.0000
2,Education,Education,1.0000
3,Escort Education,Education,1.0000
4,Shopping,Retail,1.0000
5,Other Escort,Employment,0.1479
6,Other Escort,Retail,0.1352
7,Other Escort,Healthcare,0.1164
8,Other Escort,Personal Services,0.1255
9,Other Escort,Public Services,0.1354
